In [1]:
import pandas as pd
import re

# Load diagnostics CSV
df = pd.read_csv("diagnostics.csv")

In [2]:
# df.head(10)

In [3]:
COLUMN_MAP = {
    "Title": "title",
    "Name": "raw_name",
    "Sex": "sex",
    "Folder No": "file_no",
    "Account Status": "account_status",
    "Date of Birth": "date_of_birth",
    "Age": "age",
    "MobileNo": "phone",
    "Reg. Date": "registration_date",
    "email": "email",
    "Data Entry": "registered_by"
}

df = df.rename(columns=COLUMN_MAP)



In [4]:
df.head(3)

,title,raw_name,sex,file_no,account_status,date_of_birth,age,phone,registration_date,email,registered_by
0,Ms.,"\t\t\t\t\t\t\tGALADIMA\t\t, Aisha",Female,23/01573,MEDICAID RADIOLOGY,30-Aug-88,37,80,30-Aug-23,NaN,molade aduragbemisola folakemi
1,Mr.,"(AC), A Musa",Male,24/02307,Private,18-Oct-24,1,80,18-Oct-24,NaN,molade aduragbemisola folakemi
2,Mr.,"(MERCY HEALTH), Benjamin",Male,24/02570,Private,21-Nov-24,1,80,21-Nov-24,NaN,molade aduragbemisola folakemi


In [5]:
def clean_and_split_name(name):
    if pd.isna(name):
        return "", "", ""

    # Convert to string and remove tabs, quotes, excessive spaces
    raw = str(name)
    raw = raw.replace("\t", " ")
    raw = raw.replace('"', "")
    raw = re.sub(r'\s+', ' ', raw).strip()

    full_name = raw  # preserve cleaned original

    # If name is empty or just a comma
    if raw in {",", ""}:
        return full_name, "", ""

    # Remove leading brackets or institution codes
    cleaned = re.sub(r'^[\(\[].*?[\)\]]\s*,?', '', raw)

    # Remove numeric prefixes like "002 ", "164079/"
    cleaned = re.sub(r'^\d+[/\s]*', '', cleaned).strip()

    # Final split logic
    if ',' in cleaned:
        last, first = [p.strip() for p in cleaned.split(',', 1)]
    else:
        parts = cleaned.split()
        if len(parts) == 1:
            first = ""
            last = parts[0]
        else:
            first = parts[0]
            last = " ".join(parts[1:])

    return full_name.title(), first.title(), last.title()


df["full_name"], df["first_name"], df["last_name"] = zip(
    *df["raw_name"].apply(clean_and_split_name)
)


C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\2719764989.py:38: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["full_name"], df["first_name"], df["last_name"] = zip(
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\2719764989.py:38: F

In [6]:
# df.head(10)

In [7]:
def clean_and_split_name(name):
    if pd.isna(name):
        return "", "", ""

    # Normalize raw string
    raw = str(name)
    raw = raw.replace("\t", " ")
    raw = raw.replace('"', "")
    raw = re.sub(r'\s+', ' ', raw).strip()

    full_name = raw  # preserve original cleaned name

    # Handle empty or comma-only
    if raw in {",", ""}:
        return full_name, "", ""

    # Remove leading institutions in brackets (even if unclosed)
    cleaned = re.sub(r'^[\(\[].*?[\)\]]?\s*,?', '', raw).strip()

    # Remove numeric prefixes like "002 ", "164079/"
    cleaned = re.sub(r'^\d+[/\s]*', '', cleaned).strip()

    first = ""
    last = ""

    if ',' in cleaned:
        last, first = [p.strip() for p in cleaned.split(',', 1)]
    else:
        parts = cleaned.split()
        if len(parts) == 1:
            first = parts[0]
            last = ""
        else:
            first = parts[0]
            last = " ".join(parts[1:])

    # 🧠 FIX: Handle single-letter initials like "A Musa"
    if first and len(first) == 1 and last:
        first, last = last, first

    return full_name.title(), first.title(), last.title()


In [8]:
# df.head(10)

In [9]:
# ============================================================
# FINAL NORMALIZATION & SCHEMA-SAFE ENFORCEMENT
# Guarantees compatibility with patients table NOT NULL rules
# ============================================================

# ----------------------------
# SEX (NEVER NULL)
# ----------------------------
df["sex"] = (
    df["sex"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["sex"] = df["sex"].map({
    "male": "Male",
    "female": "Female"
}).fillna("Unknown")


# ----------------------------
# PHONE (NEVER NULL)
# ----------------------------
df["phone"] = (
    df["phone"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

df["phone"] = df["phone"].apply(
    lambda x: f"+234{x[-10:]}" if len(x) >= 10 else "NA"
)


# ----------------------------
# EMAIL (nullable allowed)
# ----------------------------
df["email"] = df["email"].astype(str).str.lower().str.strip()
df.loc[df["email"] == "nan", "email"] = None


# ----------------------------
# DATES
# ----------------------------
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")
df["registration_date"] = pd.to_datetime(df["registration_date"], errors="coerce")


# ----------------------------
# NAMES (NEVER NULL)
# ----------------------------
df["first_name"] = df["first_name"].fillna("").astype(str).str.strip()
df["last_name"] = df["last_name"].fillna("").astype(str).str.strip()

# First name fallback
df.loc[df["first_name"] == "", "first_name"] = (
    df["last_name"]
    .where(df["last_name"] != "", df["full_name"].astype(str).str.split().str[0])
)

# Last name fallback
df.loc[df["last_name"] == "", "last_name"] = (
    df["first_name"]
    .where(df["first_name"] != "", "Unknown")
)


# ----------------------------
# PATIENT ID (MATCHES EXISTING SYSTEM)
# ----------------------------
df["patient_id"] = "PAT-" + df["file_no"].astype(str)


# ----------------------------
# TIMESTAMPS (NEVER NULL)
# ----------------------------
now = pd.Timestamp.now()

df["created_at"] = pd.to_datetime(
    df["registration_date"], errors="coerce"
).fillna(now)

df["updated_at"] = df["created_at"]


# ----------------------------
# FINAL SAFETY CHECK
# ----------------------------
critical_cols = [
    "patient_id",
    "first_name",
    "last_name",
    "sex",
    "phone",
    "created_at",
    "updated_at"
]

print("NULL CHECK (MUST ALL BE ZERO):")
print(df[critical_cols].isna().sum())


C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\1065247794.py:9: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["sex"] = (
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\1065247794.py:16: FutureWarning: ChainedAssignmentError: behavio

NULL CHECK (MUST ALL BE ZERO):
patient_id    0
first_name    0
last_name     0
sex           0
phone         0
created_at    0
updated_at    0
dtype: int64


C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\1065247794.py:47: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["registration_date"] = pd.to_datetime(df["registration_date"], errors="coerce")
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_

In [10]:
df.head(5)

,title,raw_name,sex,file_no,account_status,date_of_birth,age,phone,registration_date,email,registered_by,full_name,first_name,last_name,patient_id,created_at,updated_at
0,Ms.,"\t\t\t\t\t\t\tGALADIMA\t\t, Aisha",Female,23/01573,MEDICAID RADIOLOGY,1988-08-30,37,NA,2023-08-30,None,molade aduragbemisola folakemi,"Galadima , Aisha",Aisha,Galadima,PAT-23/01573,2023-08-30,2023-08-30
1,Mr.,"(AC), A Musa",Male,24/02307,Private,2024-10-18,1,NA,2024-10-18,None,molade aduragbemisola folakemi,"(Ac), A Musa",A,Musa,PAT-24/02307,2024-10-18,2024-10-18
2,Mr.,"(MERCY HEALTH), Benjamin",Male,24/02570,Private,2024-11-21,1,NA,2024-11-21,None,molade aduragbemisola folakemi,"(Mercy Health), Benjamin",Benjamin,Benjamin,PAT-24/02570,2024-11-21,2024-11-21
3,Mr.,"(MERCY HEALTH), Emeka",Male,24/02569,Private,2024-11-21,1,NA,2024-11-21,None,molade aduragbemisola folakemi,"(Mercy Health), Emeka",Emeka,Emeka,PAT-24/02569,2024-11-21,2024-11-21
4,Mr.,"(WELLINTON) ANTHONY, Osagie",Male,20/00198,Private,2057-11-12,68,NA,2020-02-12,None,"ADOGAH, Prosper","(Wellinton) Anthony, Osagie",Osagie,Anthony,PAT-20/00198,2020-02-12,2020-02-12


In [11]:
# ----------------------------
# REQUIRED / DERIVED COLUMNS
# ----------------------------

# patient_id MUST match existing hospital format
df["patient_id"] = "PAT-" + df["file_no"].astype(str)

# Optional / nullable columns
df["occupation"] = None
df["address"] = None
df["referred_by_id"] = None

# Static values
df["category"] = "diagnostics"
df["is_test"] = False


C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\1646602754.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["patient_id"] = "PAT-" + df["file_no"].astype(str)
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\1646602754.py:9: FutureW

In [12]:
df[["patient_id", "created_at", "updated_at"]].head()


,patient_id,created_at,updated_at
0,PAT-23/01573,2023-08-30,2023-08-30
1,PAT-24/02307,2024-10-18,2024-10-18
2,PAT-24/02570,2024-11-21,2024-11-21
3,PAT-24/02569,2024-11-21,2024-11-21
4,PAT-20/00198,2020-02-12,2020-02-12


In [13]:
df["patient_id"].isna().sum()   # MUST be 0


np.int64(0)

In [14]:
df.head(3)

,title,raw_name,sex,file_no,account_status,date_of_birth,age,phone,registration_date,email,...,first_name,last_name,patient_id,created_at,updated_at,occupation,address,referred_by_id,category,is_test
0,Ms.,"\t\t\t\t\t\t\tGALADIMA\t\t, Aisha",Female,23/01573,MEDICAID RADIOLOGY,1988-08-30,37,NA,2023-08-30,None,...,Aisha,Galadima,PAT-23/01573,2023-08-30,2023-08-30,None,None,None,diagnostics,False
1,Mr.,"(AC), A Musa",Male,24/02307,Private,2024-10-18,1,NA,2024-10-18,None,...,A,Musa,PAT-24/02307,2024-10-18,2024-10-18,None,None,None,diagnostics,False
2,Mr.,"(MERCY HEALTH), Benjamin",Male,24/02570,Private,2024-11-21,1,NA,2024-11-21,None,...,Benjamin,Benjamin,PAT-24/02570,2024-11-21,2024-11-21,None,None,None,diagnostics,False


In [15]:
# ----------------------------
# CREATED_AT & UPDATED_AT (REQUIRED)
# ----------------------------
now = pd.Timestamp.now()

df["created_at"] = pd.to_datetime(
    df["registration_date"], errors="coerce"
).fillna(now)

df["updated_at"] = df["created_at"]


C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\2507856608.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["created_at"] = pd.to_datetime(
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_14340\2507856608.py:10: FutureWarning: ChainedAss

In [16]:
FINAL_COLUMNS = [
    "file_no",
    "patient_id",
    "title",
    "full_name",
    "first_name",
    "last_name",
    "date_of_birth",
    "age",
    "sex",
    "occupation",
    "phone",
    "email",
    "address",
    "referred_by_id",
    "registered_by",
    "account_status",
    "registration_date",
    "category",
    "is_test",
]

df_final = df[FINAL_COLUMNS]


In [17]:
df_final.head(10)

,file_no,patient_id,title,full_name,first_name,last_name,date_of_birth,age,sex,occupation,phone,email,address,referred_by_id,registered_by,account_status,registration_date,category,is_test
0,23/01573,PAT-23/01573,Ms.,"Galadima , Aisha",Aisha,Galadima,1988-08-30,37,Female,None,NA,None,None,None,molade aduragbemisola folakemi,MEDICAID RADIOLOGY,2023-08-30,diagnostics,False
1,24/02307,PAT-24/02307,Mr.,"(Ac), A Musa",A,Musa,2024-10-18,1,Male,None,NA,None,None,None,molade aduragbemisola folakemi,Private,2024-10-18,diagnostics,False
2,24/02570,PAT-24/02570,Mr.,"(Mercy Health), Benjamin",Benjamin,Benjamin,2024-11-21,1,Male,None,NA,None,None,None,molade aduragbemisola folakemi,Private,2024-11-21,diagnostics,False
3,24/02569,PAT-24/02569,Mr.,"(Mercy Health), Emeka",Emeka,Emeka,2024-11-21,1,Male,None,NA,None,None,None,molade aduragbemisola folakemi,Private,2024-11-21,diagnostics,False
4,20/00198,PAT-20/00198,Mr.,"(Wellinton) Anthony, Osagie",Osagie,Anthony,2057-11-12,68,Male,None,NA,None,None,None,"ADOGAH, Prosper",Private,2020-02-12,diagnostics,False
5,22/05289,PAT-22/05289,Mrs.,",",",",",",2022-06-21,3,Unknown,None,NA,None,None,None,123,CODE 5,2022-06-21,diagnostics,False
6,22/01008,PAT-22/01008,Mr.,"[Pishchyk, Mikalai",Mikalai,[Pishchyk,1983-12-04,42,Male,None,+2349087937580,None,None,None,123,STAYHEALTH SURE,2022-01-19,diagnostics,False
7,20/02047,PAT-20/02047,Mr.,"002 Isreal, John",John,Isreal,2020-12-23,5,Male,None,NA,None,None,None,"IFEANYICHUKWU, Johnclinton",Private,2020-12-23,diagnostics,False
8,20/02048,PAT-20/02048,Mr.,"003 Jerry, Adikum",Adikum,Jerry,2020-12-23,5,Male,None,NA,None,None,None,"IFEANYICHUKWU, Johnclinton",Private,2020-12-23,diagnostics,False
9,20/02050,PAT-20/02050,Mr.,"004 Thankgod, Smith",Smith,Thankgod,2020-12-01,5,Male,None,NA,None,None,None,"IFEANYICHUKWU, Johnclinton",Private,2020-12-23,diagnostics,False


In [20]:
critical = [
    "patient_id",
    "first_name",
    "last_name",
    "sex",
    "phone",
]

df_final[critical].isna().sum()


patient_id    0
first_name    0
last_name     0
sex           0
phone         0
dtype: int64

In [21]:
df_final.to_csv("diagnostics_clean.csv", index=False)